In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments,T5ForConditionalGeneration


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


In [3]:
train_data.head()


,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]


"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.sample(10)


,id,dialogue,summary
8694,13680795,Reid: has pollard lost his job?\r\nLennon: I d...,Pollard lost his job. His work place is closin...
330,13865452,Pedro: I got promoted!\nSamantha: Great! Well ...,Pedro got promoted.
12752,13731048,Jane: And?\r\nJane: Have you had your scan?\r\...,"Mia had her scan done. It's growing, but she w..."
1284,13716438,Chris: Thought I should have a wander around S...,Chris went to Sawyer's and found out the price...
14527,13813467,"John: Have you seen today's news?\r\nAnn: No, ...",John saw some videos from Mike's concert in to...
588,13828660-1,Anna: do u know where my red brush is?\r\nAnna...,"Anna can't find her red brush, and Mary saw it..."
13069,13716935,"Francesca: girls, I need your advice\r\nBlake:...",Francesca is stressed as Brian wants them to g...
12480,13828678,Lisa: I love you\r\nJean-Luc: I love you too :...,Lisa and Jean-Luc love each other.
1663,13716261,"Barbara: girls, what do you think?\r\nBarbara:...",Barbara should buy it. She could also buy red ...
2059,13821224,Abigail: <file_video>\r\nAbigail: Daily portio...,Abigail tries to cheer up Olivia and Kate with...


In [6]:
train_data.shape


(14732, 3)

In [7]:
val_data.shape


(818, 3)

In [8]:
# Random Sampling
train_data = train_data.sample(n=train_data.shape[0], random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=val_data.shape[0], random_state=42).reset_index(drop=True)


In [9]:
train_data.shape


(14732, 3)

# Data Pre-processing


In [13]:
train_data = train_data.dropna(subset=["dialogue", "summary"]).reset_index(drop=True)
val_data = val_data.dropna(subset=["dialogue", "summary"]).reset_index(drop=True)

In [17]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # Lines
    text = re.sub(r"\s+", " ", text) # Spaces
    text = re.sub(r"<.*?>", " ", text) # HTML Tags
    text = text.strip().lower()
    return text

In [18]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


# Tokenize


In [19]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [20]:
# Raw Data => Tokenized Input for Fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs


In [21]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()


In [22]:
train_dataset[0]
# input ids - dialogue => token ids
# 1 => EOS, 0 => Padding
# attention mask
# labels - target => summary


{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [23]:
len(train_dataset[0]["input_ids"])


512

In [24]:
print(type(train_dataset))
print(type(val_dataset))


<class 'list'>
<class 'list'>


# Working with our Model


In [26]:
# NLP => Generation Task

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [27]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
model.to(device)


Device: mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [28]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500 # 0 => learning rate is to default
)


In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


In [30]:
# Train the Model
trainer.train()


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [23]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2631.06it/s]


# Test the core logic for summarization


In [28]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) # Clean

  # Tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # Generate the Summary => Token IDs
  model.to(device)
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # Token IDs Convert to Summary => Decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)
  return summary


In [31]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

In [32]:
summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary, "\n")

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security and long-term societal impact. 

